In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import files as colab_files  # type: ignore
except Exception:
    colab_files = None

CONFIG = {
    "stage1_core_file": "stage1_pair_results_core_filtered.csv",
    "stage1_highconv_file": "stage1_pair_results_high_conviction.csv",
    "stage2_useful_file": "stage2_regime_results_useful.csv",
    "stage2_useful_post_file": "stage2_regime_results_useful_with_post_analysis.csv",
    "stage2_presentation_file": "stage2_regime_results_presentation_subset.csv",
    "stage3_nodes_all_subsets_file": "stage3_network_nodes_all_subsets.csv",
    "stage3_edges_all_subsets_file": "stage3_network_edges_all_subsets.csv",
    "highconv_baseline": {
        "actual_aligned_obs_min": 300,
        "avg_remaining_maturity_diff_max": 1.0,
        "spread_change_corr_min": 0.40,
        "trace_margin_95_min": 5.0,
        "ect_adf_pvalue_max": 0.01,
        "half_life_min_days": 1.0,
        "half_life_max_days": 10.0,
    },
    "post_regime_baseline": {
        "min_regime_occupancy": 0.15,
        "variance_ratio": 1.50,
        "phi_gap": 0.10,
        "max_abs_lambda": 1.0,
    },
    "stage1_sensitivity": {
        "actual_aligned_obs_min": [250, 300, 350],
        "avg_remaining_maturity_diff_max": [0.75, 1.0, 1.25],
        "spread_change_corr_min": [0.35, 0.40, 0.45],
        "trace_margin_95_min": [3.0, 5.0, 7.0],
        "ect_adf_pvalue_max": [0.005, 0.01, 0.02],
        "half_life_windows": [(0.5, 10.0), (1.0, 10.0), (1.0, 15.0)],
    },
    "stage2_sensitivity": {
        "min_regime_occupancy": [0.10, 0.15, 0.20],
        "variance_ratio": [1.25, 1.50, 2.00],
        "phi_gap": [0.05, 0.10, 0.15],
    },
    "network_top_k": [10, 20],
    "output_dir": "robustness_module_outputs",
    "zip_basename": "robustness_module_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}


def _choose_col(df: pd.DataFrame, candidates, required: bool = True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the candidate columns exist: {candidates}")
    return None


def _make_pair_key(df: pd.DataFrame) -> pd.Series:
    cols_cusip = [c for c in ["cusip_1", "cusip_2"] if c in df.columns]
    if len(cols_cusip) == 2:
        left = df[cols_cusip[0]].astype(str)
        right = df[cols_cusip[1]].astype(str)
        a = np.where(left <= right, left, right)
        b = np.where(left <= right, right, left)
        return pd.Series(a + "||" + b, index=df.index)

    cols_issuer = [c for c in ["company_symbol_1", "company_symbol_2"] if c in df.columns]
    if len(cols_issuer) == 2:
        left = df[cols_issuer[0]].astype(str)
        right = df[cols_issuer[1]].astype(str)
        a = np.where(left <= right, left, right)
        b = np.where(left <= right, right, left)
        return pd.Series(a + "||" + b, index=df.index)

    raise KeyError("Could not construct pair key")


def _sector_combo(df: pd.DataFrame) -> pd.Series:
    if "sector_combo" in df.columns:
        return df["sector_combo"].astype(str)

    if "sector_1" in df.columns and "sector_2" in df.columns:
        s1 = df["sector_1"].astype(str)
        s2 = df["sector_2"].astype(str)
        a = np.where(s1 <= s2, s1, s2)
        b = np.where(s1 <= s2, s2, s1)
        return pd.Series(a + " | " + b, index=df.index)

    return pd.Series(["unknown"] * len(df), index=df.index)


def _issuer_count(df: pd.DataFrame):
    cols = [c for c in ["company_symbol_1", "company_symbol_2"] if c in df.columns]
    if len(cols) != 2:
        return np.nan
    return pd.unique(pd.concat([df[cols[0]], df[cols[1]]], ignore_index=True)).size


def _passes_highconv(df: pd.DataFrame, cfg: dict) -> pd.Series:
    return (
        (df["actual_aligned_obs"] >= cfg["actual_aligned_obs_min"])
        & (df["avg_remaining_maturity_diff"] <= cfg["avg_remaining_maturity_diff_max"])
        & (df["spread_change_corr_shortlist"] >= cfg["spread_change_corr_min"])
        & (df["trace_margin_95"] >= cfg["trace_margin_95_min"])
        & (df["ect_adf_pvalue"] <= cfg["ect_adf_pvalue_max"])
        & (df["half_life_days"] >= cfg["half_life_min_days"])
        & (df["half_life_days"] <= cfg["half_life_max_days"])
    )


def _passes_post_regime(df: pd.DataFrame, cfg: dict) -> pd.Series:
    lam0 = _choose_col(df, ["lambda_regime0", "lambda_0", "lambda0"], required=False)
    lam1 = _choose_col(df, ["lambda_regime1", "lambda_1", "lambda1"], required=False)

    cond = (
        (df["min_regime_occupancy"] >= cfg["min_regime_occupancy"])
        & (df["variance_ratio"] >= cfg["variance_ratio"])
        & (df["phi_gap"] >= cfg["phi_gap"])
    )

    if lam0 is not None:
        cond = cond & (df[lam0].abs() <= cfg["max_abs_lambda"])
    if lam1 is not None:
        cond = cond & (df[lam1].abs() <= cfg["max_abs_lambda"])

    return cond


def _summary_row(label: str, subset: pd.DataFrame) -> dict:
    out = {
        "scenario": label,
        "pairs": int(len(subset)),
        "unique_issuers": _issuer_count(subset),
        "sector_combos": int(_sector_combo(subset).nunique()),
    }

    for c in [
        "trace_margin_95",
        "actual_aligned_obs",
        "spread_change_corr_shortlist",
        "avg_remaining_maturity_diff",
        "half_life_days",
        "ect_adf_pvalue",
        "min_regime_occupancy",
        "variance_ratio",
        "phi_gap",
    ]:
        if c in subset.columns:
            out[f"median_{c}"] = float(subset[c].median()) if len(subset) else np.nan

    return out


def stage1_threshold_sensitivity(core_df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    baseline = CONFIG["highconv_baseline"]
    rows = [_summary_row("baseline", core_df.loc[_passes_highconv(core_df, baseline)])]

    grids = CONFIG["stage1_sensitivity"]

    for val in grids["actual_aligned_obs_min"]:
        cfg = baseline.copy()
        cfg["actual_aligned_obs_min"] = val
        rows.append(_summary_row(f"actual_aligned_obs_min={val}", core_df.loc[_passes_highconv(core_df, cfg)]))

    for val in grids["avg_remaining_maturity_diff_max"]:
        cfg = baseline.copy()
        cfg["avg_remaining_maturity_diff_max"] = val
        rows.append(_summary_row(f"avg_remaining_maturity_diff_max={val}", core_df.loc[_passes_highconv(core_df, cfg)]))

    for val in grids["spread_change_corr_min"]:
        cfg = baseline.copy()
        cfg["spread_change_corr_min"] = val
        rows.append(_summary_row(f"spread_change_corr_min={val}", core_df.loc[_passes_highconv(core_df, cfg)]))

    for val in grids["trace_margin_95_min"]:
        cfg = baseline.copy()
        cfg["trace_margin_95_min"] = val
        rows.append(_summary_row(f"trace_margin_95_min={val}", core_df.loc[_passes_highconv(core_df, cfg)]))

    for val in grids["ect_adf_pvalue_max"]:
        cfg = baseline.copy()
        cfg["ect_adf_pvalue_max"] = val
        rows.append(_summary_row(f"ect_adf_pvalue_max={val}", core_df.loc[_passes_highconv(core_df, cfg)]))

    for lo, hi in grids["half_life_windows"]:
        cfg = baseline.copy()
        cfg["half_life_min_days"] = lo
        cfg["half_life_max_days"] = hi
        rows.append(_summary_row(f"half_life_window=[{lo},{hi}]", core_df.loc[_passes_highconv(core_df, cfg)]))

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "stage1_threshold_sensitivity.csv", index=False)
    return df


def stage2_threshold_sensitivity(useful_df: pd.DataFrame, out_dir: Path) -> pd.DataFrame:
    baseline = CONFIG["post_regime_baseline"]
    rows = [_summary_row("baseline", useful_df.loc[_passes_post_regime(useful_df, baseline)])]

    grids = CONFIG["stage2_sensitivity"]

    for val in grids["min_regime_occupancy"]:
        cfg = baseline.copy()
        cfg["min_regime_occupancy"] = val
        rows.append(_summary_row(f"min_regime_occupancy={val}", useful_df.loc[_passes_post_regime(useful_df, cfg)]))

    for val in grids["variance_ratio"]:
        cfg = baseline.copy()
        cfg["variance_ratio"] = val
        rows.append(_summary_row(f"variance_ratio={val}", useful_df.loc[_passes_post_regime(useful_df, cfg)]))

    for val in grids["phi_gap"]:
        cfg = baseline.copy()
        cfg["phi_gap"] = val
        rows.append(_summary_row(f"phi_gap={val}", useful_df.loc[_passes_post_regime(useful_df, cfg)]))

    df = pd.DataFrame(rows)
    df.to_csv(out_dir / "stage2_regime_threshold_sensitivity.csv", index=False)
    return df


def _subset_frames(useful_post_df: pd.DataFrame, presentation_df: pd.DataFrame) -> dict:
    frames = {"useful_all": useful_post_df.copy()}

    if "high_quality_useful" in useful_post_df.columns:
        frames["high_quality"] = useful_post_df.loc[useful_post_df["high_quality_useful"].fillna(False)].copy()
    else:
        frames["high_quality"] = useful_post_df.copy()

    temp = useful_post_df.copy()
    temp["_pair_key"] = _make_pair_key(temp)
    pres_keys = set(_make_pair_key(presentation_df)) if len(presentation_df) else set()
    frames["presentation"] = temp.loc[temp["_pair_key"].isin(pres_keys)].drop(columns=["_pair_key"])

    return frames


def _edge_weight_baseline(df: pd.DataFrame) -> pd.Series:
    if "post_regime_rank_score" in df.columns:
        return df["post_regime_rank_score"].astype(float)

    comps = []
    for c in ["trace_margin_95", "variance_ratio", "phi_gap", "min_regime_occupancy"]:
        if c in df.columns:
            s = df[c].astype(float)
            comps.append((s - s.mean()) / (s.std(ddof=0) + 1e-9))

    return sum(comps) / len(comps) if comps else pd.Series(np.ones(len(df)), index=df.index)


def _edge_weight_simple(df: pd.DataFrame) -> pd.Series:
    vr = df["variance_ratio"].clip(upper=8.0) if "variance_ratio" in df.columns else 1.0
    pg = df["phi_gap"] if "phi_gap" in df.columns else 1.0
    occ = df["min_regime_occupancy"] if "min_regime_occupancy" in df.columns else 1.0

    return (
        pd.Series(vr, index=df.index).astype(float) * 0.5
        + pd.Series(pg, index=df.index).astype(float) * 2.0
        + pd.Series(occ, index=df.index).astype(float) * 2.0
    )


def _build_node_ranking(df: pd.DataFrame, weight: pd.Series) -> pd.DataFrame:
    c1 = _choose_col(df, ["company_symbol_1"])
    c2 = _choose_col(df, ["company_symbol_2"])

    tmp = pd.DataFrame(
        {
            "i1": df[c1].astype(str),
            "i2": df[c2].astype(str),
            "w": weight.astype(float),
        }
    )

    agg = {}
    for _, row in tmp.iterrows():
        agg[row["i1"]] = agg.get(row["i1"], 0.0) + row["w"]
        agg[row["i2"]] = agg.get(row["i2"], 0.0) + row["w"]

    return (
        pd.DataFrame({"issuer": list(agg.keys()), "weighted_degree": list(agg.values())})
        .sort_values("weighted_degree", ascending=False)
        .reset_index(drop=True)
    )


def network_robustness(useful_post_df: pd.DataFrame, presentation_df: pd.DataFrame, out_dir: Path):
    frames = _subset_frames(useful_post_df, presentation_df)
    top_ks = CONFIG["network_top_k"]

    node_rows = []
    overlap_rows = []
    rankings = {}

    for subset_name, subset_df in frames.items():
        if len(subset_df) == 0:
            continue

        base_rank = _build_node_ranking(subset_df, _edge_weight_baseline(subset_df))
        simp_rank = _build_node_ranking(subset_df, _edge_weight_simple(subset_df))

        rankings[(subset_name, "baseline")] = base_rank
        rankings[(subset_name, "simple")] = simp_rank

        node_rows.append(
            {
                "subset": subset_name,
                "weight_scheme": "baseline",
                "n_nodes": len(base_rank),
                "top5": ", ".join(base_rank["issuer"].head(5).astype(str).tolist()),
            }
        )
        node_rows.append(
            {
                "subset": subset_name,
                "weight_scheme": "simple",
                "n_nodes": len(simp_rank),
                "top5": ", ".join(simp_rank["issuer"].head(5).astype(str).tolist()),
            }
        )

        for k in top_ks:
            b = set(base_rank["issuer"].head(k))
            s = set(simp_rank["issuer"].head(k))
            overlap_rows.append(
                {
                    "comparison": f"{subset_name}: baseline vs simple",
                    "top_k": k,
                    "overlap_count": len(b & s),
                    "overlap_share": len(b & s) / k if k else np.nan,
                }
            )

    subset_names = ["useful_all", "high_quality", "presentation"]
    for i in range(len(subset_names)):
        for j in range(i + 1, len(subset_names)):
            a_name = subset_names[i]
            b_name = subset_names[j]
            if (a_name, "baseline") not in rankings or (b_name, "baseline") not in rankings:
                continue

            a_rank = rankings[(a_name, "baseline")]
            b_rank = rankings[(b_name, "baseline")]

            for k in top_ks:
                a = set(a_rank["issuer"].head(k))
                b = set(b_rank["issuer"].head(k))
                overlap_rows.append(
                    {
                        "comparison": f"{a_name} vs {b_name}: baseline",
                        "top_k": k,
                        "overlap_count": len(a & b),
                        "overlap_share": len(a & b) / k if k else np.nan,
                    }
                )

    node_df = pd.DataFrame(node_rows)
    overlap_df = pd.DataFrame(overlap_rows)

    node_df.to_csv(out_dir / "network_robustness_node_summary.csv", index=False)
    overlap_df.to_csv(out_dir / "network_robustness_overlap_summary.csv", index=False)

    return node_df, overlap_df


def compile_report(stage1_sens, stage2_sens, net_overlap, out_dir: Path):
    rows = []

    rows.append(
        {
            "section": "stage1_highconv_thresholds",
            "headline": "baseline_highconv_pairs",
            "value": int(stage1_sens.loc[stage1_sens["scenario"] == "baseline", "pairs"].iloc[0]),
        }
    )
    rows.append(
        {
            "section": "stage2_regime_thresholds",
            "headline": "baseline_high_quality_useful_pairs",
            "value": int(stage2_sens.loc[stage2_sens["scenario"] == "baseline", "pairs"].iloc[0]),
        }
    )

    for _, row in net_overlap.iterrows():
        rows.append(
            {
                "section": "network_overlap",
                "headline": f'{row["comparison"]} top{int(row["top_k"])} overlap_share',
                "value": float(row["overlap_share"]),
            }
        )

    report = pd.DataFrame(rows)
    report.to_csv(out_dir / "robustness_module_report.csv", index=False)
    return report


def write_readme(out_dir: Path):
    text = """Robustness module outputs
=========================

This module is a fourth standalone step that reads outputs from the
three model notebooks/scripts rather than combining those notebooks.

Main checks included:
1. Stage 1 threshold sensitivity around the high-conviction filter
2. Stage 2 threshold sensitivity around the post-regime high-quality filter
3. Network robustness using alternative weight schemes and subset overlap
"""
    (out_dir / "README_robustness_module.txt").write_text(text)


def zip_output_dir(out_dir: Path, zip_basename: str) -> Path:
    zip_path = out_dir / f"{zip_basename}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in out_dir.rglob("*"):
            if file_path == zip_path or file_path.is_dir():
                continue
            zf.write(file_path, arcname=file_path.relative_to(out_dir))
    return zip_path


def main():
    out_dir = Path(CONFIG["output_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    core_df = pd.read_csv(CONFIG["stage1_core_file"])
    useful_df = pd.read_csv(CONFIG["stage2_useful_file"])
    useful_post_df = pd.read_csv(CONFIG["stage2_useful_post_file"])
    presentation_df = pd.read_csv(CONFIG["stage2_presentation_file"])

    stage1_sens = stage1_threshold_sensitivity(core_df, out_dir)
    stage2_sens = stage2_threshold_sensitivity(useful_df, out_dir)
    _, net_overlap = network_robustness(useful_post_df, presentation_df, out_dir)
    compile_report(stage1_sens, stage2_sens, net_overlap, out_dir)

    (out_dir / "robustness_module_config.json").write_text(json.dumps(CONFIG, indent=2))
    write_readme(out_dir)

    if CONFIG.get("zip_outputs", True):
        zip_path = zip_output_dir(out_dir, CONFIG["zip_basename"])
        print(f"Created zip: {zip_path}")
        if CONFIG.get("auto_download_zip_if_colab", True) and colab_files is not None:
            try:
                colab_files.download(str(zip_path))
            except Exception as e:
                print(f"Colab download failed: {e}")


if __name__ == "__main__":
    main()

Created zip: robustness_module_outputs/robustness_module_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>